In [2]:
import sqlite3
import pandas as pd 

connection = sqlite3.connect("../data/movies.db")

movies = pd.read_sql_query("SELECT * FROM movies", connection)
genres = pd.read_sql_query("SELECT * FROM genres", connection)
movie_genres = pd.read_sql_query("SELECT * FROM movie_genres", connection)
people = pd.read_sql_query("SELECT * FROM people", connection)
movie_credits = pd.read_sql_query("SELECT * FROM movie_credits", connection)

In [22]:
links = pd.read_csv("../data/raw/ml-latest-small/links.csv")
all_ids = set(links['tmdbId'].dropna())
saved_ids = set(movies['tmdb_id'].dropna())
missing_ids = all_ids - saved_ids
missing_movies =  links[links['tmdbId'].isin(missing_ids)]
missing_movies

,movieId,imdbId,tmdbId
3127,4207,100232,12773.0
3362,4568,96913,17882.0
3680,5069,270933,68149.0
3741,5209,81809,24549.0
4981,7646,259153,14980.0
...,...,...,...
9603,176329,6254796,466986.0
9634,179135,6769208,463612.0
9651,180263,118460,106035.0
9690,184257,5189670,502892.0


In [100]:
len(saved_ids)/len(all_ids)

0.9883900133566218

In [ ]:
# make movies from csv to analize missing movies

In [35]:
movies[movies[['title', 'release_date']].duplicated(keep=False)].sort_values('title')

,tmdb_id,title,release_date,runtime,budget,revenue,vote_average,vote_count


In [ ]:
movie_with_genres = movie_genres.merge(
    movies, how='left', on='tmdb_id').merge(
        genres, how='left', right_on='id', left_on='genre_id')

genre_stats = movie_with_genres.groupby('genre').agg(
    average_rating=('vote_average', 'mean'),
    review_count=('vote_count', 'sum'),
    movie_count=('tmdb_id', 'count')
).round(2).sort_values('review_count', ascending=False)
genre_stats

,average_rating,review_count,movie_count
genre,,,
Drama,6.73,6549879,4474
Action,6.34,5752388,1846
Comedy,6.33,5199889,3918
Adventure,6.47,4982650,1355
Thriller,6.42,4333236,1963
Science Fiction,6.34,3635467,1064
Crime,6.57,2811366,1482
Fantasy,6.50,2656625,894
Romance,6.50,2372141,1933


In [88]:
movie_with_cast = movies.merge(
    movie_credits, how='left', on='tmdb_id').merge(
        people, how='left', on='person_id')

cast_stats = movie_with_cast.groupby(['name', 'role']).agg(
        average_rating=('vote_average', 'mean'),
    review_count=('vote_count', 'sum'),
    movie_count=('tmdb_id', 'count')
).round(2).sort_values('review_count', ascending=False)
cast_stats 
cast_stats.reset_index('role', inplace=True)
cast_stats[cast_stats['role'] == 'Director']
cast_stats[cast_stats['role'] == 'Actor']

/tmp/ipykernel_21483/475244396.py:11: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  cast_stats.reset_index('role', inplace=True)


,role,average_rating,review_count,movie_count
name,,,,
Johnny Depp,Actor,6.72,286132,51
Robert Downey Jr.,Actor,6.60,280738,41
Tom Hanks,Actor,6.92,280735,43
Leonardo DiCaprio,Actor,7.24,278913,26
Brad Pitt,Actor,7.01,275834,34
...,...,...,...,...
Rollyn Stafford,Actor,6.00,1,1
Amber Arrigotti,Actor,6.00,1,1
Brian O'Shaughnessy,Actor,7.00,1,1


In [93]:
movies['profit'] = movies['revenue'] - movies['budget']
movies.sort_values('profit', ascending=False)

,tmdb_id,title,release_date,runtime,budget,revenue,vote_average,vote_count,profit
5326,19995,Avatar,2009-12-16,162,237000000,2923706026,7.611,35032,2686706026
392,597,Titanic,1997-12-18,194,200000000,2264162353,7.901,28068,2064162353
8509,140607,Star Wars: The Force Awakens,2015-12-15,136,245000000,2068223624,7.237,21135,1823223624
9139,299536,Avengers: Infinity War,2018-04-25,149,300000000,2052415039,8.247,33398,1752415039
8484,135397,Jurassic World,2015-06-06,124,150000000,1671537444,6.702,22114,1521537444
...,...,...,...,...,...,...,...,...,...
9158,302429,Strange Magic,2015-01-23,99,100000000,13603453,7.300,609,-86396547
9504,400106,Bright,2017-12-22,117,90000000,0,6.162,6157,-90000000
2525,9822,Mighty Joe Young,1998-12-25,114,90000000,0,6.315,912,-90000000
3632,11692,The Adventures of Pluto Nash,2002-08-16,95,100000000,7103973,4.381,565,-92896027
